# 26. Masked pose pretraining: a graph encoder with a Transformer decoder (validation only)

**The idea.** Pretrain the ST-GCN encoder on the badminton pose clips themselves, without labels, by hiding parts of each clip and asking a
small Transformer decoder to reconstruct them. Then discard the decoder and fine tune the encoder for shot classification. The combination
is a hybrid: the graph convolutions (the best performing family here) build the representation, and a Transformer decoder, which can attend
across every joint and frame at once, does the reconstruction.

**Why this, from our own results.**
- Notebook 25 (leave-group-out) found the information redundant across the body: removing any one joint group costs the same, so a hidden joint
  can be inferred from the others. That is exactly what masked reconstruction exploits.
- Joint masking is already part of the advanced augmentation family that helped, so masking is a familiar, non-destructive operation here.
- NTU60 pretraining (notebook 21) did not help, plausibly because of the mismatch (different layout, 10 blocks, different cameras and actions).
  Pretraining in domain removes that mismatch.
- Every supervised model side change so far was null, so the remaining untested lever is the quality of the representation learned from data.

**Pretraining data.** Training clips only, plus the fixed synthetic clips already built from them (basic family). Validation and test clips are
never used for pretraining. Basic augmentation is applied while pretraining (rotation, jitter, mirror, crop) so the target is always the
augmented clip itself. Masks are redrawn every epoch: random spans of 8 to 15 frames on about a third of the joints, plus a block of 4 to 8
whole frames on half the clips. Masked entries are zeroed (position, confidence and bone), the same convention as the joint masking augmentation.

**Fine tuning arms.** All use the premise arm's data: classes built up to 500 with synthetic clips, advanced online augmentation, natural
frequencies, the tuned recipe in `configs/stgcn_recipe.json`.

| arm | initialisation | backbone learning rate |
|---|---|---|
| `scratch` (control) | random | 1.0 x |
| `pretrained` | masked pretrained backbone, new head | 1.0 x |
| `pretrained_lowlr` | masked pretrained backbone, new head | 0.2 x (head 1.0 x) |

**Rule, fixed before running.** Validation only, tuning seeds 77 and 88 (one pretraining run per seed), checkpoints chosen on the composite
(mean of validation top-1 and MCA). A pretrained arm passes if its mean validation top-1 is at least **0.5 points above scratch** and its
mean validation MCA is **not more than 1 point below**. The test set is never evaluated. A pass is a reason to run reporting seeds, not a result.

**Sanity check on the pretraining itself.** The decoder's reconstruction error on validation clips is compared with two trivial predictors
(everything at the hip, and each joint's average over its visible frames). If the model does not beat the second one, it learned no motion.

Logs to `BadmintonPoseAnalysis/experiments/masked_pose_pretraining`. One pretraining run takes roughly 15 to 25 minutes on the pod.

In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from clearml import Task
from sklearn.metrics import precision_recall_fscore_support

REPO = os.environ.get("BADMINTON_REPO", "/workspace/badminton-honours-project")
DATA = os.environ.get("BADMINTON_DATA", "/workspace/data")
OUTPUTS = os.environ.get("BADMINTON_OUTPUTS", "/workspace/outputs")
CHECKPOINTS = os.environ.get("BADMINTON_CHECKPOINTS", "/workspace/checkpoints")
sys.path.insert(0, REPO)
import dataset as ds
from models import stgcn as stgcn_model
import training as tr

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__, device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
PROJECT = "BadmintonPoseAnalysis/experiments/masked_pose_pretraining"
KEYPOINTS = f"{DATA}/keypoints_interp"
OFFLINE_ROOT = f"{DATA}/keypoints_offline_aug"
OUT = f"{OUTPUTS}/experiments/masked_pose_pretraining"
CKPT = f"{CHECKPOINTS}/masked_pose"
os.makedirs(OUT, exist_ok=True)
os.makedirs(CKPT, exist_ok=True)

SEEDS = [77, 88]
SPLITS = ("train", "val")                      # the test set is never evaluated here
SELECT_METRIC = "composite"
NORMALISATION, USE_COURT = "body", True
NUM_WORKERS, EVAL_WORKERS = 20, 8
OFFLINE_TARGET, RARE_THRESHOLD = 500, 150
MIN_GAIN_TOP1, MAX_MCA_DROP = 0.005, 0.010     # the rule, as points / 100

PRETRAIN = dict(epochs=100, batch_size=64, lr=1e-3, weight_decay=0.05, grad_clip=1.0,
                joint_p=0.35, span=(8, 15), block_p=0.5, block=(4, 8),
                dec_dim=128, dec_layers=2, dec_heads=4, augmentation="basic")
FINETUNE_ARMS = {"scratch": None, "pretrained": 1.0, "pretrained_lowlr": 0.2}   # value: backbone lr multiplier

with open(f"{REPO}/configs/stgcn_recipe.json") as f:
    RECIPE = json.load(f)

manifest = ds.load_filtered_manifest(f"{DATA}/split_manifest.csv", f"{DATA}/extraction_log.csv",
                                     f"{DATA}/keypoints_interp_log.csv", RECIPE["min_native_frames"])
class_names = sorted(manifest["class_name"].unique())
class_to_id = {c: i for i, c in enumerate(class_names)}
K = len(class_names)
short = [c.split("_", 1)[1] for c in class_names]
split_hash = open(f"{DATA}/split_manifest.sha256").read().split()[0]
train_counts = manifest[manifest["split"] == "train"]["class_name"].value_counts().reindex(class_names)
rare_ids = [i for i, n in enumerate(train_counts.to_numpy()) if n < RARE_THRESHOLD]

synthetic = ds.build_offline_augmented_set(manifest, KEYPOINTS, OFFLINE_ROOT, target=OFFLINE_TARGET, seed=0, policy="basic")
manifest_offline = pd.concat([manifest, synthetic], ignore_index=True)
assert synthetic["parent"].isin(manifest[manifest["split"] == "train"]["filepath"]).all()
IN_CH = stgcn_model.n_channels(USE_COURT)
T_FRAMES, V = 30, ds.NUM_JOINTS
print(f"pretraining and fine tuning set: {int((manifest_offline['split'] == 'train').sum())} training clips "
      f"({len(synthetic)} synthetic); {len(rare_ids)} limited sample classes; input channels {IN_CH}")

## Model: masking, encoder input and decoder

The encoder is exactly the ST-GCN backbone used everywhere else (9 blocks, 64 to 256 channels, mmaction2). Its output is a grid of
256 channel features over 8 time steps by 17 joints. The decoder turns each of those 8 x 17 tokens into the coordinates of its joint for the
4 original frames it covers, after two Transformer layers over all 136 tokens (learned position embeddings), so a token can use any other
joint or time step to fill in what was hidden.

In [ ]:
class PoseDecoder(nn.Module):
    # tokens (N, C, T', V) -> reconstructed x, y for every frame and joint, (N, 4 T', V, 2)
    def __init__(self, in_ch=256, dim=128, layers=2, heads=4, t_tokens=8, v=17, frames_per_token=4):
        super().__init__()
        self.t_tokens, self.frames_per_token = t_tokens, frames_per_token
        self.proj = nn.Linear(in_ch, dim)
        self.pos = nn.Parameter(torch.zeros(1, t_tokens * v, dim))
        nn.init.trunc_normal_(self.pos, std=0.02)
        layer = nn.TransformerEncoderLayer(dim, heads, dim * 2, dropout=0.0, batch_first=True, norm_first=True)
        self.tf = nn.TransformerEncoder(layer, layers)
        self.out = nn.Linear(dim, frames_per_token * 2)

    def forward(self, f):
        n, c, t, v = f.shape
        assert t == self.t_tokens, f"expected {self.t_tokens} time steps from the encoder, got {t}"
        x = self.tf(self.proj(f.permute(0, 2, 3, 1).reshape(n, t * v, c)) + self.pos)
        y = self.out(x).view(n, t, v, self.frames_per_token, 2).permute(0, 1, 3, 2, 4)
        return y.reshape(n, t * self.frames_per_token, v, 2)


class MaskedPose(nn.Module):
    def __init__(self, backbone, decoder):
        super().__init__()
        self.backbone, self.decoder = backbone, decoder

    def forward(self, x):                      # x (N, 1, T, V, C)
        return self.decoder(self.backbone(x)[:, 0])


def make_mask(n, gen, p=PRETRAIN):
    # True where a joint at a frame is hidden: joint spans plus a block of whole frames, shape (N, T, V)
    t, v = T_FRAMES, V
    ar = torch.arange(t).view(1, t, 1)
    has = torch.rand(n, 1, v, generator=gen) < p["joint_p"]
    length = torch.randint(p["span"][0], p["span"][1] + 1, (n, 1, v), generator=gen)
    start = (torch.rand(n, 1, v, generator=gen) * (t - length + 1)).long()
    spans = (ar >= start) & (ar < start + length) & has
    blen = torch.randint(p["block"][0], p["block"][1] + 1, (n, 1, 1), generator=gen)
    bstart = (torch.rand(n, 1, 1, generator=gen) * (t - blen + 1)).long()
    use_block = torch.rand(n, 1, 1, generator=gen) < p["block_p"]
    blocks = (ar >= bstart) & (ar < bstart + blen) & use_block
    return spans | blocks


PARENT_T = torch.as_tensor(ds.PARENT, dtype=torch.long)


def encoder_input(batch, mask=None):
    kp, bone = batch["keypoints"].to(device).float(), batch["bone"].to(device).float()
    if mask is not None:
        m = mask.to(device)
        # a bone is joint minus parent, so a hidden joint is recoverable from the bones of its children; hide every bone that touches one
        bone_hidden = m | m[:, :, PARENT_T.to(device)]
        kp, bone = kp.masked_fill(m[..., None], 0.0), bone.masked_fill(bone_hidden[..., None], 0.0)
    court = batch["court"].to(device).float()[:, :, None, :].expand(-1, -1, V, -1)
    return torch.cat([kp, bone, court], dim=-1)[:, None]            # (N, 1 person, T, V, C)


def masked_error(pred_xy, batch, mask):
    # smooth L1 summed over x, y, returned as (sum over hidden entries, number of hidden entries)
    target = batch["keypoints"][..., :2].to(device).float()
    m = mask.to(device).float()
    err = nn.functional.smooth_l1_loss(pred_xy[:, :target.shape[1]], target, reduction="none", beta=0.1).sum(-1)
    return (err * m).sum(), m.sum()


def static_prediction(batch, mask):
    # each joint's average position over its visible frames in the same clip
    target = batch["keypoints"][..., :2].to(device).float()
    vis = (~mask.to(device)).float()[..., None]
    mean = (target * vis).sum(1, keepdim=True) / vis.sum(1, keepdim=True).clamp(min=1)
    return mean.expand_as(target)


@torch.no_grad()
def validate(net, loader, seed=1234):
    net.eval()
    gen = torch.Generator().manual_seed(seed)
    tot = {"model": 0.0, "zero": 0.0, "static": 0.0}
    count = 0.0
    for batch in loader:
        mask = make_mask(len(batch["label"]), gen)
        pred = net(encoder_input(batch, mask))
        zero = torch.zeros_like(batch["keypoints"][..., :2]).to(device)
        for name, p in (("model", pred), ("zero", zero), ("static", static_prediction(batch, mask))):
            s, c = masked_error(p, batch, mask)
            tot[name] += s.item()
        count += c.item()
    return {k: v / count for k, v in tot.items()}

## Smoke test

The mask hides a sensible share of entries and never a whole joint, one pretraining step runs forwards and backwards, the backbone weights
save and load into a fresh classifier with exactly matching tensors, and a fine tuning step runs. Run this cell alone first.

In [ ]:
tr.seed_everything(0)
loaders = tr.build_loaders(manifest_offline, KEYPOINTS, class_to_id, PRETRAIN["augmentation"], NORMALISATION, seed=0,
                           batch_size=PRETRAIN["batch_size"], num_workers=2, eval_workers=2, balanced=False)
batch = next(iter(loaders["train"]))
gen = torch.Generator().manual_seed(0)
mask = make_mask(len(batch["label"]), gen)
frac = mask.float().mean().item()
assert 0.1 < frac < 0.6, frac
x_chk = encoder_input(batch, mask)[:, 0]                                  # (N, T, V, C); channels 3 and 4 are the bone
assert x_chk[..., 3:5][mask.to(device)[:, :, PARENT_T.to(device)]].abs().sum() == 0, "a bone that touches a hidden joint is still visible (leak)"
assert x_chk[..., :3][mask.to(device)].abs().sum() == 0
assert (mask.sum(1) < T_FRAMES).all(), "a joint is hidden in every frame"
print(f"hidden share of joint frames: {frac:.2f}")

base = stgcn_model.build_stgcn(IN_CH, K)
net = MaskedPose(base.backbone, PoseDecoder(dim=PRETRAIN["dec_dim"], layers=PRETRAIN["dec_layers"], heads=PRETRAIN["dec_heads"])).to(device)
x = encoder_input(batch, mask)
assert x.shape[1:] == (1, T_FRAMES, V, IN_CH), x.shape
feat = net.backbone(x)
print("encoder output", tuple(feat.shape))
pred = net(x)
assert pred.shape[0] == x.shape[0] and pred.shape[2:] == (V, 2) and pred.shape[1] >= T_FRAMES, pred.shape
s, c = masked_error(pred, batch, mask)
loss = s / c
loss.backward()
assert all(p.grad is not None for p in net.backbone.parameters() if p.requires_grad), "no gradient reached the encoder"
print(f"reconstruction loss at initialisation {loss.item():.4f}; parameters: encoder "
      f"{sum(p.numel() for p in net.backbone.parameters()):,}, decoder {sum(p.numel() for p in net.decoder.parameters()):,}")

# transfer round trip
path = f"{CKPT}/_smoke_backbone.pt"
torch.save(net.backbone.state_dict(), path)
clf = stgcn_model.build_stgcn(IN_CH, K)
clf.backbone.load_state_dict(torch.load(path, map_location="cpu"))
for (k1, v1), (k2, v2) in zip(net.backbone.state_dict().items(), clf.backbone.state_dict().items()):
    assert k1 == k2 and torch.equal(v1.cpu(), v2), k1
os.remove(path)

forward = stgcn_model.make_forward(USE_COURT)
clf = clf.to(device)
logits = forward(clf, batch, device)
assert logits.shape == (len(batch["label"]), K)
nn.CrossEntropyLoss()(logits, batch["label"].to(device)).backward()
print("transfer round trip and fine tuning step ok; smoke test passed")
del net, base, clf, loaders

## Pretraining (one run per tuning seed)

Training clips and their synthetic extensions only. The decoder is thrown away afterwards; only the encoder weights are kept.

In [ ]:
def pretrain(seed):
    path, summary_path = f"{CKPT}/backbone_s{seed}.pt", f"{OUT}/mpt_pretrain_s{seed}_summary.json"
    if os.path.exists(path) and os.path.exists(summary_path):
        print(f"pretraining seed {seed} already done")
        return json.load(open(summary_path))
    run_name = f"mpt_pretrain_s{seed}"
    tr.seed_everything(seed)
    task = Task.init(project_name=PROJECT, task_name=run_name, task_type=Task.TaskTypes.training,
                     tags=["masked_pose", "pretraining", f"seed{seed}"], reuse_last_task_id=False,
                     auto_connect_frameworks={"matplotlib": False, "pytorch": False})
    task.connect({**{k: (list(v) if isinstance(v, tuple) else v) for k, v in PRETRAIN.items()}, "seed": seed, "split_sha256": split_hash, "pretrain_clips": int((manifest_offline["split"] == "train").sum())})
    logger = task.get_logger()
    loaders = tr.build_loaders(manifest_offline, KEYPOINTS, class_to_id, PRETRAIN["augmentation"], NORMALISATION, seed=seed,
                               batch_size=PRETRAIN["batch_size"], num_workers=NUM_WORKERS, eval_workers=EVAL_WORKERS, balanced=False)
    base = stgcn_model.build_stgcn(IN_CH, K)
    net = MaskedPose(base.backbone, PoseDecoder(dim=PRETRAIN["dec_dim"], layers=PRETRAIN["dec_layers"], heads=PRETRAIN["dec_heads"])).to(device)
    opt = torch.optim.AdamW(net.parameters(), lr=PRETRAIN["lr"], weight_decay=PRETRAIN["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=PRETRAIN["epochs"])
    gen = torch.Generator().manual_seed(seed)
    history = []
    for epoch in range(PRETRAIN["epochs"]):
        net.train()
        total, n = 0.0, 0.0
        for batch in loaders["train"]:
            mask = make_mask(len(batch["label"]), gen)
            s, c = masked_error(net(encoder_input(batch, mask)), batch, mask)
            loss = s / c
            if not torch.isfinite(loss):
                raise FloatingPointError(f"non finite loss at epoch {epoch}")
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), PRETRAIN["grad_clip"])
            opt.step()
            total, n = total + s.item(), n + c.item()
        sched.step()
        val = validate(net, loaders["val"])
        row = {"epoch": epoch, "train_masked_loss": total / n, **{f"val_{k}": v for k, v in val.items()}}
        history.append(row)
        logger.report_scalar("masked reconstruction", "train", row["train_masked_loss"], iteration=epoch)
        for k in ("model", "zero", "static"):
            logger.report_scalar("masked reconstruction", f"val {k}", row[f"val_{k}"], iteration=epoch)
        if epoch % 10 == 0 or epoch == PRETRAIN["epochs"] - 1:
            print(f"epoch {epoch:3d}  train {row['train_masked_loss']:.4f}  val {row['val_model']:.4f}  "
                  f"(hip only {row['val_zero']:.4f}, visible frame average {row['val_static']:.4f})")
    torch.save(net.backbone.state_dict(), path)
    pd.DataFrame(history).to_csv(f"{OUT}/{run_name}_history.csv", index=False)
    summary = {"seed": seed, **{k: float(history[-1][f"val_{k}"]) for k in ("model", "zero", "static")},
               "train_final": float(history[-1]["train_masked_loss"])}
    json.dump(summary, open(summary_path, "w"), indent=2)
    for k, v in summary.items():
        if k != "seed":
            logger.report_single_value(k, v)
    task.close()
    del net, base, loaders
    torch.cuda.empty_cache()
    return summary


pretrain_summaries = [pretrain(s) for s in SEEDS]
pd.DataFrame(pretrain_summaries).rename(columns={"model": "val: decoder", "zero": "val: hip only", "static": "val: visible frame average",
                                                 "train_final": "train: decoder"})

## Fine tuning

Same recipe for all arms, only the initialisation and the backbone learning rate differ. A run that already exists in `results.csv` is skipped.

In [ ]:
results_path = f"{OUT}/results.csv"
done = pd.read_csv(results_path) if os.path.exists(results_path) else pd.DataFrame(columns=["arm", "seed"])
forward = stgcn_model.make_forward(USE_COURT)

for seed in SEEDS:
    for arm, mult in FINETUNE_ARMS.items():
        if ((done["arm"] == arm) & (done["seed"] == seed)).any():
            print(f"skip {arm} seed {seed}")
            continue
        run_name = f"mpt_{arm}_s{seed}"
        print(f"\n=== {run_name} (backbone lr x {mult}) ===")
        tr.seed_everything(seed)
        task = Task.init(project_name=PROJECT, task_name=run_name, task_type=Task.TaskTypes.training,
                         tags=["masked_pose", "finetune", arm, f"seed{seed}"], reuse_last_task_id=False,
                         auto_connect_frameworks={"matplotlib": False, "pytorch": False})
        task.connect({**RECIPE, "arm": arm, "seed": seed, "backbone_lr_multiplier": mult, "select_metric": SELECT_METRIC,
                      "balanced": False, "policy": "advanced", "offline_target": OFFLINE_TARGET, "split_sha256": split_hash})
        loaders = tr.build_loaders(manifest_offline, KEYPOINTS, class_to_id, "advanced", NORMALISATION, seed=seed,
                                   batch_size=RECIPE["batch_size"], target_per_class=RECIPE["target_per_class"],
                                   num_workers=NUM_WORKERS, eval_workers=EVAL_WORKERS, balanced=False)
        model = stgcn_model.build_stgcn(IN_CH, K).to(device)
        if mult is None:
            groups = model.parameters()
        else:
            model.backbone.load_state_dict(torch.load(f"{CKPT}/backbone_s{seed}.pt", map_location=device))
            groups = [{"params": list(model.backbone.parameters()), "lr": RECIPE["lr"] * mult},
                      {"params": list(model.cls_head.parameters()), "lr": RECIPE["lr"]}]
        optimizer = torch.optim.SGD(groups, lr=RECIPE["lr"], momentum=RECIPE["momentum"], weight_decay=RECIPE["weight_decay"])
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=RECIPE["gamma"])
        criterion = nn.CrossEntropyLoss(label_smoothing=RECIPE["label_smoothing"])
        _, history, best_epoch = tr.train_model(model, forward, loaders, optimizer, scheduler, criterion, device,
                                                task.get_logger(), num_epochs=RECIPE["num_epochs"],
                                                patience=RECIPE.get("patience", 20), select_metric=SELECT_METRIC, splits=SPLITS)
        metrics = tr.report_run(task, model, forward, loaders, class_names, device, history, best_epoch,
                                out_dir=f"{OUT}/{run_name}", run_name=run_name, splits=SPLITS, show=False,
                                extra={"arm": arm, "seed": seed})
        task.close()
        done = pd.concat([done, pd.DataFrame([metrics])], ignore_index=True)
        done.to_csv(results_path, index=False)
        del model, loaders
        torch.cuda.empty_cache()

## Results (validation only)

In [ ]:
res = pd.read_csv(results_path)
res["val_composite"] = 0.5 * (res["val_top1"] + res["val_mca"])


def val_group_recall(arm, seed):
    p = pd.read_csv(f"{OUT}/mpt_{arm}_s{seed}/mpt_{arm}_s{seed}_predictions.csv")
    p = p[p["split"] == "val"]
    y, yh = p["label"].map(class_to_id).to_numpy(), p["pred"].map(class_to_id).to_numpy()
    _, r, f1, _ = precision_recall_fscore_support(y, yh, labels=list(range(K)), zero_division=0)
    return r[rare_ids].mean(), f1.mean()


res["val_limited_recall"], res["val_macro_f1_check"] = zip(*[val_group_recall(a, s) for a, s in zip(res["arm"], res["seed"])])
cols = ["val_top1", "val_mca", "val_composite", "val_limited_recall", "train_top1", "best_epoch"]
per_seed = res[["arm", "seed"] + cols].sort_values(["arm", "seed"])
mean = res.groupby("arm")[cols].mean().reindex(list(FINETUNE_ARMS))
std = res.groupby("arm")[cols].std().reindex(list(FINETUNE_ARMS))
show = mean.copy()
for c in cols:
    if c != "best_epoch":
        show[c] = (100 * mean[c]).round(2)
    show[c + " sd"] = std[c].mul(100 if c != "best_epoch" else 1).round(2)
print("mean over seeds", SEEDS, "(points; sd is the spread between the two seeds)")
show[[c for c in show.columns if not c.endswith(" sd")]]

In [ ]:
per_seed.assign(**{c: (100 * per_seed[c]).round(2) for c in cols if c != "best_epoch"})

In [ ]:
control = mean.loc["scratch"]
rows = []
for arm in ("pretrained", "pretrained_lowlr"):
    d_top1 = mean.loc[arm, "val_top1"] - control["val_top1"]
    d_mca = mean.loc[arm, "val_mca"] - control["val_mca"]
    seed_gain = [res[(res.arm == arm) & (res.seed == s)]["val_top1"].iloc[0] - res[(res.arm == "scratch") & (res.seed == s)]["val_top1"].iloc[0] for s in SEEDS]
    passed = bool(d_top1 >= MIN_GAIN_TOP1 and d_mca >= -MAX_MCA_DROP)
    rows.append({"arm": arm, "top1 vs scratch (pts)": round(100 * d_top1, 2), "mca vs scratch (pts)": round(100 * d_mca, 2),
                 "composite vs scratch (pts)": round(100 * (mean.loc[arm, "val_composite"] - control["val_composite"]), 2),
                 "limited sample recall vs scratch (pts)": round(100 * (mean.loc[arm, "val_limited_recall"] - control["val_limited_recall"]), 2),
                 "top1 gain per seed": [round(100 * g, 2) for g in seed_gain], "passes rule": passed})
rule = pd.DataFrame(rows)
passing = [r["arm"] for r in rows if r["passes rule"]]
selected = max(passing, key=lambda a: mean.loc[a, "val_composite"]) if passing else None
print("rule: mean val top1 >= +0.5 and mean val MCA not more than 1.0 below scratch")
print("selected:", selected or "none, pretraining is not reported as helping")
rule

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for seed, c in zip(SEEDS, ("#4c72b0", "#dd8452")):
    h = pd.read_csv(f"{OUT}/mpt_pretrain_s{seed}_history.csv")
    axes[0].plot(h["epoch"], h["val_model"], color=c, label=f"decoder, seed {seed}")
    axes[0].axhline(h["val_static"].iloc[-1], color=c, ls="--", lw=0.8)
    axes[0].axhline(h["val_zero"].iloc[-1], color=c, ls=":", lw=0.8)
axes[0].set_xlabel("pretraining epoch"); axes[0].set_ylabel("validation masked error")
axes[0].set_title("Reconstruction vs trivial predictors\n(dashed: visible frame average, dotted: hip only)")
axes[0].legend(frameon=False)
for arm, c in zip(FINETUNE_ARMS, ("#b0b0b0", "#4c72b0", "#dd8452")):
    for seed, ls in zip(SEEDS, ("-", "--")):
        h = pd.read_csv(f"{OUT}/mpt_{arm}_s{seed}/mpt_{arm}_s{seed}_history.csv")
        axes[1].plot(h["epoch"], 100 * h["val_composite"], color=c, ls=ls, label=f"{arm} s{seed}")
axes[1].set_xlabel("fine tuning epoch"); axes[1].set_ylabel("validation composite (points)")
axes[1].set_title("Fine tuning, composite of val top-1 and MCA"); axes[1].legend(frameon=False, fontsize=7, ncol=2)
fig.tight_layout()
plt.show()

In [ ]:
decision = {"experiment": "masked_pose_pretraining", "seeds": SEEDS, "select_metric": SELECT_METRIC,
            "rule": {"min_gain_top1": MIN_GAIN_TOP1, "max_mca_drop": MAX_MCA_DROP},
            "pretrain": {k: (list(v) if isinstance(v, tuple) else v) for k, v in PRETRAIN.items()},
            "pretrain_val_error": pretrain_summaries,
            "arms_mean": {a: {c: float(mean.loc[a, c]) for c in cols} for a in FINETUNE_ARMS},
            "selected": selected,
            "mean_best_epoch": float(mean.loc[selected, "best_epoch"]) if selected else None}
os.makedirs(f"{REPO}/configs", exist_ok=True)
with open(f"{REPO}/configs/masked_pretraining.json", "w") as f:
    json.dump(decision, f, indent=2)

task = Task.init(project_name=PROJECT, task_name="masked_pose_summary", task_type=Task.TaskTypes.qc,
                 tags=["masked_pose", "summary"], reuse_last_task_id=False, auto_connect_frameworks={"matplotlib": False})
log = task.get_logger()
task.connect({**RECIPE, "seeds": SEEDS, "select_metric": SELECT_METRIC, "split_sha256": split_hash})
log.report_table("validation summary", "val", iteration=0, table_plot=show.reset_index())
log.report_table("rule", "val", iteration=0, table_plot=rule.astype({"top1 gain per seed": str}))
log.report_matplotlib_figure("pretraining and fine tuning", "val", figure=fig, iteration=0, report_interactive=False)
task.upload_artifact("decision", artifact_object=f"{REPO}/configs/masked_pretraining.json")
task.close()
print("saved configs/masked_pretraining.json")

## How to read this

- **First, did the pretraining learn anything?** The decoder's validation error should sit clearly below the visible frame average (the dashed line).
  If it does not, the representation holds no motion beyond a static pose and fine tuning cannot be expected to gain.
- **Then the rule.** With two seeds and 781 validation clips, a gain under about 1 point of top-1 is within noise; the rule asks for 0.5 so a pass is
  a reason to run reporting seeds (3 to 5, test set, scratch against the selected arm), not evidence.
- **`pretrained` against `pretrained_lowlr`.** If only the low learning rate arm helps, the pretrained weights are useful but fragile to the full learning rate.
- **If nothing passes**, that is a result: in domain masked pretraining on 6k clips does not help, consistent with the input limited reading of the earlier
  experiments, and goes in the write-up with the reconstruction check.
- **Limits.** One pretraining run per seed, one masking recipe, one decoder size; none of them were tuned. A near miss would justify one tuning round
  (mask ratio, epochs), a clear null would not.